# SSL400 Master Training Notebook
## Step-by-step guide. Run each cell ONE AT A TIME and wait for it to finish!
---
| EXP | Enhancement | MixUp | Augmentation |
|-----|-------------|-------|--------------|
| 1   | None (Baseline) | ❌ OFF | ❌ OFF |
| 2   | CLAHE + Gamma   | ✅ ON  | ✅ ON  |
| 3   | Bilateral + Unsharp | ✅ ON | ✅ ON |
| 4   | Hybrid (All)    | ✅ ON  | ✅ ON  |

In [ ]:
# ============================================================
# CELL 1: INSTALL DEPENDENCIES
# ============================================================
!pip install tf-keras ultralytics --quiet
print('✅ Dependencies Installed!')

In [ ]:
# ============================================================
# CELL 2: COPY PROJECT FILES FROM DATASET
# ============================================================
import os
os.makedirs('/kaggle/working/data/processed', exist_ok=True)
os.makedirs('/kaggle/working/models', exist_ok=True)
os.makedirs('/kaggle/working/logs', exist_ok=True)

# 1. Copy RAW videos from the old folder
old_base = '/kaggle/input/datasets/shenithchanidu/ssl400-dataset-v2/ssl400_full_kaggle_upload/'
os.system(f'cp -r {old_base}data/raw /kaggle/working/data/')

# 2. Copy New Code and Splits from the patch folder
patch_base = '/kaggle/input/datasets/shenithchanidu/ssl400-dataset-v2/kaggle_patch_v3/'
os.system(f'cp -r {patch_base}src /kaggle/working/')
os.system(f'cp -r {patch_base}data/splits /kaggle/working/data/')
os.system(f'cp {patch_base}config.yaml /kaggle/working/')

# Verify files
print('Split files found:', os.listdir('/kaggle/working/data/splits'))
print('✅ All project files copied successfully!')
else:
    print('❌ ERROR: Files did not copy properly.')

In [ ]:
# ============================================================
# CELL 3: APPLY ALL BUG FIXES (Run ONCE at the start of session)
# ============================================================

# --- FIX 1: EfficientNetV2 Normalization Fix ---
# EfficientNetV2 expects pixels in [0, 255].
# Our .npy files are saved as [-1, 1].
# This fix converts them back to [0, 255] AFTER all augmentation.
file_path = '/kaggle/working/src/data/tf_dataset_builder.py'
with open(file_path, 'r') as f:
    code = f.read()

# Only patch if not already patched
if '_unnormalize' not in code:
    fix = '''
    # CRITICAL BUG FIX: Un-normalize for EfficientNetV2
    # EfficientNetV2 expects inputs in [0, 255]. Our data is in [-1, 1].
    def _unnormalize(frames, labels):
        frames = (frames + 1.0) * 127.5
        return frames, labels
    
    ds = ds.map(_unnormalize, num_parallel_calls=AUTOTUNE)

    ds = ds.prefetch(AUTOTUNE)

    return ds
'''
    code = code.replace('    ds = ds.prefetch(AUTOTUNE)\n\n    return ds', fix)
    with open(file_path, 'w') as f:
        f.write(code)
    print('✅ FIX 1: Normalization patch applied!')
else:
    print('✅ FIX 1: Normalization patch already applied, skipping.')

# --- FIX 2: MixUp OFF for EXP1 Baseline, ON for EXP2-4 ---
# EXP1 has use_augmentation=False in config.yaml
# EXP2-4 have use_augmentation=True in config.yaml
# This patch makes MixUp automatically follow the use_augmentation flag.
train_path = '/kaggle/working/src/training/train.py'
with open(train_path, 'r') as f:
    train_code = f.read()

if 'exp.get("use_augmentation"' not in train_code:
    train_code = train_code.replace(
        'use_mixup=(p1["mixup_alpha"] > 0),',
        'use_mixup=(p1["mixup_alpha"] > 0 and exp.get("use_augmentation", True)),'
    )
    train_code = train_code.replace(
        'use_mixup=(p2["mixup_alpha"] > 0),',
        'use_mixup=(p2["mixup_alpha"] > 0 and exp.get("use_augmentation", True)),'
    )
    with open(train_path, 'w') as f:
        f.write(train_code)
    print('✅ FIX 2: MixUp conditional patch applied!')
    print('   → EXP1 (use_augmentation=False): MixUp will be OFF')
    print('   → EXP2-4 (use_augmentation=True): MixUp will be ON')
else:
    print('✅ FIX 2: MixUp patch already applied, skipping.')

print('\n🎯 All fixes applied! Ready to train.')

In [ ]:
# ============================================================
# CELL 4: SET EXPERIMENT ID
# Change this number to run each experiment:
#   EXP_ID = 1  →  Baseline (No Enhancement, No MixUp)
#   EXP_ID = 2  →  CLAHE + Gamma + MixUp
#   EXP_ID = 3  →  Bilateral + Unsharp + MixUp
#   EXP_ID = 4  →  Hybrid + MixUp
# ============================================================
EXP_ID = 1

import yaml
with open('/kaggle/working/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
exp = config['experiments'][EXP_ID - 1]
print(f'\n{"="*50}')
print(f'  Running: EXP {EXP_ID} — {exp["name"]}')
print(f'  Augmentation: {exp["use_augmentation"]}')
print(f'  MixUp: {"ON" if exp["use_augmentation"] else "OFF"}')
print(f'{"="*50}\n')

In [ ]:
# ============================================================
# CELL 5: PROCESS VIDEOS INTO ENHANCED FRAMES
# This applies CLAHE/YOLO/etc. and saves .npy files
# ============================================================
!cd /kaggle/working && python src/data/video_to_frames.py --exp_id {EXP_ID}

In [ ]:
# ============================================================
# CELL 6: VERIFY DATA IS CORRECTLY LOADED
# Run this to confirm all files found before training!
# ============================================================
import pandas as pd
from pathlib import Path
import yaml

with open('/kaggle/working/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
exp = config['experiments'][EXP_ID - 1]
proc_dir = Path('/kaggle/working') / exp['processed_dir']

for split in ['train', 'val', 'test']:
    csv_path = f'/kaggle/working/data/splits/{split}_split.csv'
    df = pd.read_csv(csv_path)
    found = 0
    for _, row in df.iterrows():
        stem = Path(str(row['video_path'])).stem
        cid = int(row['class_id'])
        if (proc_dir / str(cid) / f'{stem}.npy').exists():
            found += 1
    status = '✅' if found == len(df) else '⚠️'
    print(f'{status} {split.upper()}: {found}/{len(df)} files found')
print('\n🎯 If all show ✅, you are ready to train!')

In [ ]:
# ============================================================
# CELL 7: TRAIN THE MODEL (PHASE 1 + PHASE 2)
# ============================================================
!cd /kaggle/working && python src/training/train.py --exp_id {EXP_ID}

In [ ]:
# ============================================================
# CELL 8: EVALUATE - GENERATE CLASSIFICATION REPORT
# ============================================================
import os, sys, yaml, numpy as np
import tensorflow as tf
import tf_keras as keras
sys.path.insert(0, '/kaggle/working/src')
os.chdir('/kaggle/working')

from models.efficientnet_builder import build_model
from data.tf_dataset_builder import build_dataset
from sklearn.metrics import classification_report

with open('config.yaml', 'r') as f:
    config = yaml.safe_load(f)

exp = config['experiments'][EXP_ID - 1]
test_ds = build_dataset(
    split_csv='/kaggle/working/data/splits/test_split.csv',
    processed_dir=f"/kaggle/working/{exp['processed_dir']}",
    num_classes=8, batch_size=2, is_training=False,
    num_frames=config['frames']['num_frames'],
    target_size=(config['frames']['width'], config['frames']['height'])
)

model_path = f"models/experiment_{EXP_ID}/best_model_phase2.keras"
if os.path.exists(model_path):
    model = build_model(num_classes=8, num_frames=32, img_height=224, img_width=224,
                        lstm_units=512, dropout_rate=0.4)
    model.load_weights(model_path)
    true_labels, predictions = [], []
    for frames, labels in test_ds:
        preds = model.predict(frames, verbose=0)
        true_labels.extend(np.argmax(labels.numpy(), axis=1))
        predictions.extend(np.argmax(preds, axis=1))
    label_map = {0:'Thank you', 1:'Hello', 2:'Good', 3:'House', 4:'Eat', 5:'Drink', 6:'Tell', 7:'Write'}
    print(f"\n{'='*50}\n  EXP {EXP_ID} RESULTS: {exp['name']}\n{'='*50}")
    print(classification_report(true_labels, predictions,
          target_names=[label_map[i] for i in range(8)], zero_division=0))
else:
    print(f'❌ Model not found at {model_path}. Did training complete?')

In [ ]:
# ============================================================
# CELL 9: GENERATE THESIS PLOTS
# ============================================================
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from sklearn.metrics import confusion_matrix

label_map = {0:'Thank you', 1:'Hello', 2:'Good', 3:'House', 4:'Eat', 5:'Drink', 6:'Tell', 7:'Write'}
class_names = [label_map[i] for i in range(8)]
exp_name = config['experiments'][EXP_ID-1]['name']

# --- Confusion Matrix ---
cm = confusion_matrix(true_labels, predictions)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title(f'EXP {EXP_ID}: {exp_name}\nConfusion Matrix', fontsize=13, fontweight='bold')
plt.ylabel('True Label'); plt.xlabel('Predicted Label')
plt.xticks(rotation=45, ha='right'); plt.tight_layout()
plt.savefig(f'/kaggle/working/exp{EXP_ID}_confusion_matrix.png', dpi=150)
plt.show(); plt.close()

# --- Training Curves ---
try:
    p1 = pd.read_csv(f'/kaggle/working/logs/experiment_{EXP_ID}/training_log_phase1.csv')
    p2 = pd.read_csv(f'/kaggle/working/logs/experiment_{EXP_ID}/training_log_phase2.csv')
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    offset = len(p1)
    axes[0].plot(p1['accuracy'], 'b--', label='P1 Train')
    axes[0].plot(p1['val_accuracy'], 'b-', label='P1 Val')
    axes[0].plot(range(offset, offset+len(p2)), p2['accuracy'], 'r--', label='P2 Train')
    axes[0].plot(range(offset, offset+len(p2)), p2['val_accuracy'], 'r-', label='P2 Val')
    axes[0].set_title(f'EXP {EXP_ID} Accuracy', fontweight='bold')
    axes[0].legend(); axes[0].grid(alpha=0.3)
    axes[1].plot(p1['loss'], 'b--', label='P1 Train')
    axes[1].plot(p1['val_loss'], 'b-', label='P1 Val')
    axes[1].plot(range(offset, offset+len(p2)), p2['loss'], 'r--', label='P2 Train')
    axes[1].set_title(f'EXP {EXP_ID} Loss', fontweight='bold')
    axes[1].legend(); axes[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.savefig(f'/kaggle/working/exp{EXP_ID}_training_curves.png', dpi=150)
    plt.show(); plt.close()
    print(f'✅ EXP {EXP_ID} plots saved!')
except Exception as e:
    print(f'Could not generate training curves: {e}')

In [ ]:
# ============================================================
# CELL 10: DOWNLOAD RESULTS (Click green buttons to download)
# ============================================================
import base64, os
from IPython.display import HTML, display

def make_link(fpath, text):
    if not os.path.exists(fpath):
        return f'<p style="color:red">❌ Missing: {os.path.basename(fpath)}</p>'
    with open(fpath, 'rb') as f:
        data = base64.b64encode(f.read()).decode()
    fname = os.path.basename(fpath)
    ext = fname.split('.')[-1]
    mime = 'image/png' if ext == 'png' else 'text/csv'
    return (f'<a href="data:{mime};base64,{data}" download="{fname}" '
            f'style="display:block;margin:6px;padding:10px;background:#27ae60;'
            f'color:white;border-radius:5px;text-decoration:none;font-weight:bold;">'
            f'⬇️ {text}</a>')

links = [
    make_link(f'/kaggle/working/exp{EXP_ID}_confusion_matrix.png', f'EXP{EXP_ID} Confusion Matrix'),
    make_link(f'/kaggle/working/exp{EXP_ID}_training_curves.png', f'EXP{EXP_ID} Training Curves'),
    make_link(f'/kaggle/working/logs/experiment_{EXP_ID}/training_log_phase1.csv', f'EXP{EXP_ID} Phase1 Log'),
    make_link(f'/kaggle/working/logs/experiment_{EXP_ID}/training_log_phase2.csv', f'EXP{EXP_ID} Phase2 Log'),
]
display(HTML(f'<h3>EXP {EXP_ID} Download Results:</h3>' + ''.join(links)))

In [ ]:
# ============================================================
# CELL 11: CLEANUP DISK (Run before starting next experiment!)
# This deletes processed frames to free up disk space.
# Your trained MODEL and LOGS are kept safe!
# ============================================================
import yaml
with open('/kaggle/working/config.yaml', 'r') as f:
    config = yaml.safe_load(f)
exp = config['experiments'][EXP_ID - 1]
proc_dir = f"/kaggle/working/{exp['processed_dir']}"
!rm -rf {proc_dir}
print(f'✅ Deleted: {proc_dir}')
print('Now change EXP_ID in Cell 4 and run from Cell 5!')